# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [6]:
import pandas as pd

# Load the dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Clean column names for easier access (lower case and underscores)
df.columns = [col.lower().replace(" ", "_") for col in df.columns]

# 1. Filter for low claim amount (< 1000) AND Response is 'Yes'
filtered_df = df[(df['total_claim_amount'] < 1000) & (df['response'] == 'Yes')]

print(f"Customers with low claims and 'Yes' response: {len(filtered_df)}")

# 2. Grouping by Policy Type and Gender for 'Yes' responders
responders_df = df[df['response'] == 'Yes']

analysis = responders_df.groupby(['policy_type', 'gender'])[['monthly_premium_auto', 'customer_lifetime_value', 'total_claim_amount']].mean()

print("Average Metrics for Responders:")
print(analysis)

# Discussion/Insight:
# Segments where CLV is high but Total Claim Amount is low are the 'Golden' segments.
# For example, if 'Personal Auto' females have high premiums but much lower claims 
# than males, they are a high-profit, low-risk segment.

# 3. Count customers by state and filter for states > 500
state_counts = df.groupby('state')['customer'].count()
large_states = state_counts[state_counts > 500]

print("States with more than 500 customers:")
print(large_states)

# 4. Max, Min, and Median CLV
clv_stats = df.groupby(['education', 'gender'])['customer_lifetime_value'].agg(['max', 'min', 'median'])

print("CLV Statistics by Education and Gender:")
print(clv_stats)

# Conclusions:
# Usually, higher education (Doctor/Master) shows a higher median CLV. 
# Significant gaps between Max and Min suggest that while education is a factor, 
# individual risk profiles vary wildly within the same education level.

# Convert date and extract month
df['effective_to_date'] = pd.to_datetime(df['effective_to_date'])
df['month'] = df['effective_to_date'].dt.month_name()

# 5. Pivot Table: Months as columns, States as rows
state_month_pivot = df.pivot_table(index='state', columns='month', values='customer', aggfunc='count')
print("Policies sold by State and Month:")
display(state_month_pivot)

# 6. Top 3 States Analysis
# Step A: Find top 3 states by total policy count
top_3_states = df['state'].value_counts().nlargest(3).index.tolist()

# Step B: Filter DF for only those states and group by month
top_3_df = df[df['state'].isin(top_3_states)]
top_3_analysis = top_3_df.groupby(['state', 'month'])['customer'].count().reset_index()

print(f"\nPolicy counts for top 3 states ({top_3_states}):")
print(top_3_analysis)


# 7. Response Rate by Channel
# Create a dummy column: 1 if Yes, 0 if No
df['is_yes'] = df['response'].apply(lambda x: 1 if x == 'Yes' else 0)

# Calculate rate per channel
response_rates = df.groupby('sales_channel')['is_yes'].mean().reset_index()

# Using Melt to show the response rate in a tidy format
melted_response = pd.melt(response_rates, id_vars=['sales_channel'], value_vars=['is_yes'], var_name='metric', value_name='response_rate')

print("Customer Response Rate by Marketing Channel:")
print(melted_response)



Customers with low claims and 'Yes' response: 1399
Average Metrics for Responders:
                       monthly_premium_auto  customer_lifetime_value  \
policy_type    gender                                                  
Corporate Auto F                  94.301775              7712.628736   
               M                  92.188312              7944.465414   
Personal Auto  F                  98.998148              8339.791842   
               M                  91.085821              7448.383281   
Special Auto   F                  92.314286              7691.584111   
               M                  86.343750              8247.088702   

                       total_claim_amount  
policy_type    gender                      
Corporate Auto F               433.738499  
               M               408.582459  
Personal Auto  F               452.965929  
               M               457.010178  
Special Auto   F               453.280164  
               M               4

C:\Users\pablo\AppData\Local\Temp\ipykernel_3988\2922385037.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['effective_to_date'] = pd.to_datetime(df['effective_to_date'])


month,February,January
state,,
Arizona,929,1008
California,1634,1918
Nevada,442,551
Oregon,1344,1565
Washington,425,463



Policy counts for top 3 states (['California', 'Oregon', 'Arizona']):
        state     month  customer
0     Arizona  February       929
1     Arizona   January      1008
2  California  February      1634
3  California   January      1918
4      Oregon  February      1344
5      Oregon   January      1565
Customer Response Rate by Marketing Channel:
  sales_channel  metric  response_rate
0         Agent  is_yes       0.180053
1        Branch  is_yes       0.107876
2   Call Center  is_yes       0.103223
3           Web  is_yes       0.108856
